In [0]:
%sql
CREATE VOLUME workspace.default.Inputs;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5715869600778684>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE VOLUME workspace.default.Inputs;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208     self.driver_activity_logge

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import requests
import json

input_path = "/Volumes/workspace/default/Inputs/"

In [0]:
display(dbutils.fs.ls(input_path))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/Inputs/credits_and_tags_IMDB_TMDB.csv,credits_and_tags_IMDB_TMDB.csv,22320678,1791394484000
dbfs:/Volumes/workspace/default/Inputs/movies_financials_IMDB_TMDB.csv,movies_financials_IMDB_TMDB.csv,1467743,1791394137000
dbfs:/Volumes/workspace/default/Inputs/movies_info_TMDB_IMDB.csv,movies_info_TMDB_IMDB.csv,33924063,1791394509000
dbfs:/Volumes/workspace/default/Inputs/movies_metrics_IMDB_TMDB.csv,movies_metrics_IMDB_TMDB.csv,3323884,1791394142000
dbfs:/Volumes/workspace/default/Inputs/movies_reviews.csv,movies_reviews.csv,1923431,1791394138000


In [0]:
spark.sql("""CREATE SCHEMA IF NOT EXISTS bronze""")

DataFrame[]

In [0]:
def carregar_bronze(nome_arquivo, nome_tabela):
        caminho = input_path + nome_arquivo
        
        df = (
                spark.read
                .option("header", True)
                .option("inferSchema", True)
                .option("multiLine", True)
                .option("escape", '"')
                .csv(caminho)
        )

        df = df.withColumn(
                "ingestion_datetime",
                F.current_timestamp()
        )

        (
                df.write
                .format("delta")
                .mode("append")
                .saveAsTable(nome_tabela)
        )
        
        print(f"Tabela {nome_tabela} criada/carregada com sucesso.")

In [0]:
carregar_bronze(
    "movies_info_TMDB_IMDB.csv",
    "bronze.tb_movies_info"
)

carregar_bronze(
    "movies_financials_IMDB_TMDB.csv",
    "bronze.tb_movies_financials"
)

carregar_bronze(
    "movies_metrics_IMDB_TMDB.csv",
    "bronze.tb_movies_metrics"
)

carregar_bronze(
    "credits_and_tags_IMDB_TMDB.csv",
    "bronze.tb_credits_and_tags"
)

carregar_bronze(
    "movies_reviews.csv",
    "bronze.tb_movies_reviews"
)

Tabela bronze.tb_movies_info criada/carregada com sucesso.
Tabela bronze.tb_movies_financials criada/carregada com sucesso.
Tabela bronze.tb_movies_metrics criada/carregada com sucesso.
Tabela bronze.tb_credits_and_tags criada/carregada com sucesso.
Tabela bronze.tb_movies_reviews criada/carregada com sucesso.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Cria o schema Silver
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS silver
""")

# Lê a tabela Bronze
df_info = spark.table("bronze.tb_movies_info")

# Limpeza e transformação
df_info_silver = (
    df_info
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.trim(F.col("title")).alias("titulo"),
        F.trim(F.col("original_title")).alias("titulo_original"),
        F.trim(F.col("original_language")).alias("idioma_original"),
        F.expr("try_cast(runtime AS INT)").alias("duracao_minutos"),
        F.trim(F.col("status")).alias("status"),
        F.col("overview").alias("sinopse"),
        F.col("tagline").alias("slogan"),

        # Data no formato MM-dd-yyyy
        F.expr("""
            try_to_date(release_date, 'MM-dd-yyyy')
        """).alias("data_lancamento"),

        F.col("ingestion_datetime")
    )
    .withColumn(
        "ano_lancamento",
        F.year("data_lancamento")
    )
)

# Normaliza o status
df_info_silver = df_info_silver.withColumn(
    "status",
    F.when(
        F.upper(F.trim(F.col("status"))).isin(
            "RELEASED", "LANÇADO"
        ),
        "Lançado"
    )
    .when(
        F.upper(F.trim(F.col("status"))).isin(
            "POST PRODUCTION", "PÓS-PRODUÇÃO"
        ),
        "Pós-produção"
    )
    .when(
        F.upper(F.trim(F.col("status"))).isin(
            "IN PRODUCTION", "EM PRODUÇÃO"
        ),
        "Em produção"
    )
    .when(
        F.upper(F.trim(F.col("status"))).isin(
            "PLANNED", "PLANEJADO"
        ),
        "Planejado"
    )
    .otherwise(
        F.initcap(F.trim(F.col("status")))
    )
)

# Remove registros sem ID
# e mantém somente o registro mais recente de cada filme
janela = Window.partitionBy("id_filme").orderBy(
    F.col("ingestion_datetime").desc()
)

df_info_silver = (
    df_info_silver
    .filter(F.col("id_filme").isNotNull())
    .withColumn("rn", F.row_number().over(janela))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Grava a tabela Silver
(
    df_info_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_info_filmes")
)

print("Tabela silver.tb_info_filmes criada com sucesso!")

Tabela silver.tb_info_filmes criada com sucesso!


In [0]:
df_info = spark.table("bronze.tb_movies_info")

df_info.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tconst: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
display(df_info.limit(5))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-07T18:30:15.976Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-07T18:30:15.976Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-07T18:30:15.976Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-07T18:30:15.976Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-07T18:30:15.976Z


In [0]:
df_financeiro = spark.table("bronze.tb_movies_financials")

df_financeiro.printSchema()

root
 |-- id: integer (nullable = true)
 |-- budget: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F

df_financeiro_silver = (
    df_financeiro
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.expr("try_cast(budget AS DECIMAL(20,2))").alias("orcamento"),
        F.expr("try_cast(revenue AS DECIMAL(20,2))").alias("receita"),
        F.col("ingestion_datetime")
    )
    .withColumn(
        "lucro",
        F.col("receita") - F.col("orcamento")
    )
)

(
    df_financeiro_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_financeiro_filmes")
)

print("Tabela silver.tb_financeiro_filmes criada com sucesso!")

Tabela silver.tb_financeiro_filmes criada com sucesso!


In [0]:
display(
    spark.table("silver.tb_financeiro_filmes").limit(10)
)

id_filme,orcamento,receita,ingestion_datetime,lucro
293660,58000000.00,null,2026-10-07T18:30:25.123Z,null
299536,300000000.00,2052415039.00,2026-10-07T18:30:25.123Z,1752415039.00
299534,356000000.00,2800000000.00,2026-10-07T18:30:25.123Z,2444000000.00
475557,55000000.00,1074458282.00,2026-10-07T18:30:25.123Z,1019458282.00
271110,250000000.00,null,2026-10-07T18:30:25.123Z,null
284054,200000000.00,1349926083.00,2026-10-07T18:30:25.123Z,1149926083.00
284052,180000000.00,676343174.00,2026-10-07T18:30:25.123Z,496343174.00
315635,175000000.00,880166924.00,2026-10-07T18:30:25.123Z,705166924.00
283995,200000000.00,863756051.00,2026-10-07T18:30:25.123Z,663756051.00
297761,175000000.00,746846894.00,2026-10-07T18:30:25.123Z,571846894.00


In [0]:
df_metricas = spark.table("bronze.tb_movies_metrics")

df_metricas.printSchema()

root
 |-- id: integer (nullable = true)
 |-- popularity: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- numVotes: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
display(
    spark.table("bronze.tb_movies_metrics").limit(10)
)

id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-07T20:43:36.835Z
299536,"154,34",8.255,27713,8.4,1406782,2026-10-07T20:43:36.835Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-07T20:43:36.835Z
475557,"54,522",8.168,23425,8.3,1723035,2026-10-07T20:43:36.835Z
271110,70.741,7.4,21541,7.8,947222,2026-10-07T20:43:36.835Z
284054,43.665,7.39,null,7.3,924922,2026-10-07T20:43:36.835Z
284052,70.535,7.427,20935,7.5,895880,2026-10-07T20:43:36.835Z
315635,65.88,7.345,20507,7.4,835116,2026-10-07T20:43:36.835Z
283995,67.553,7.624,20353,7.6,844767,2026-10-07T20:43:36.835Z
297761,35.356,5.909,20097,5.9,null,2026-10-07T20:43:36.835Z


In [0]:
from pyspark.sql import functions as F

# Lê a tabela Bronze
df_metricas = spark.table("bronze.tb_movies_metrics")

# Trata e transforma as métricas
df_metricas_silver = (
    df_metricas
    .select(
        F.col("id").cast("int").alias("id_filme"),

        # Popularidade: troca vírgula por ponto antes de converter
        F.expr(
            "try_cast(replace(popularity, ',', '.') as double)"
        ).alias("popularidade"),

        # Nota média do TMDB
        F.expr(
            "try_cast(replace(vote_average, ',', '.') as double)"
        ).alias("nota_media_tmdb"),

        # Quantidade de votos do TMDB
        F.expr(
            "try_cast(vote_count as int)"
        ).alias("quantidade_votos_tmdb"),

        # Nota média do IMDb
        F.expr(
            "try_cast(replace(averageRating, ',', '.') as double)"
        ).alias("nota_media_imdb"),

        # Quantidade de votos do IMDb
        F.expr(
            "try_cast(numVotes as int)"
        ).alias("quantidade_votos_imdb"),

        F.current_timestamp().alias("ingestion_datetime")
    )
)

# Grava a tabela Silver
(
    df_metricas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_metricas_engajamento")
)

print("Tabela silver.tb_metricas_engajamento criada com sucesso!")

Tabela silver.tb_metricas_engajamento criada com sucesso!


In [0]:
display(
    spark.table("silver.tb_metricas_engajamento").limit(10)
)

id_filme,popularidade,nota_media_tmdb,quantidade_votos_tmdb,nota_media_imdb,quantidade_votos_imdb,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-07T20:56:01.715Z
299536,154.34,8.255,27713,8.4,1406782,2026-10-07T20:56:01.715Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-07T20:56:01.715Z
475557,54.522,8.168,23425,8.3,1723035,2026-10-07T20:56:01.715Z
271110,70.741,7.4,21541,7.8,947222,2026-10-07T20:56:01.715Z
284054,43.665,7.39,null,7.3,924922,2026-10-07T20:56:01.715Z
284052,70.535,7.427,20935,7.5,895880,2026-10-07T20:56:01.715Z
315635,65.88,7.345,20507,7.4,835116,2026-10-07T20:56:01.715Z
283995,67.553,7.624,20353,7.6,844767,2026-10-07T20:56:01.715Z
297761,35.356,5.909,20097,5.9,null,2026-10-07T20:56:01.715Z


In [0]:
print("Tabelas do schema Silver:")

display(
    spark.sql("SHOW TABLES IN silver")
)

Tabelas do schema Silver:


database,tableName,isTemporary
silver,tb_financeiro_filmes,false
silver,tb_info_filmes,false
silver,tb_metricas_engajamento,false


In [0]:
df_creditos = spark.table("bronze.tb_credits_and_tags")

df_creditos.printSchema()

root
 |-- id: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
display(
    spark.table("bronze.tb_credits_and_tags")
    .limit(10)
)

id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-07T20:43:40.226Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-07T20:43:40.226Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-07T20:43:40.226Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-10-07T20:43:40.226Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-10-07T20:43:40.226Z
284054,"Action, Adventure, Science Fiction",Marvel Studios,United States of America,"English, Korean, Swahili, Xhosa","africa, superhero, based on comic, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Ryan Coogler,"Ryan Coogler, Joe Robert Cole, Stan Lee, Jack Kirby","Chadwick Boseman, Michael B. Jordan, Lupita Nyong'o, Danai Gurira, Martin Freeman, Daniel Kaluuya, Letitia Wright, Winston Duke, Sterling K. Brown, Angela Bassett",2026-10-07T20:43:40.226Z
284052,"Action, Adventure, Fantasy",Marvel Studios,UNITED STATES OF AMERICA,English,"magic, superhero, training, time, based on comic, sorcerer, doctor, neurosurgeon, wizard, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Scott Derrickson,N/A,"Benedict Cumberbatch, Chiwetel Ejiofor, Rachel McAdams, Benedict Wong, Mads Mikkelsen, Tilda Swinton, Michael Stuhlbarg, Benjamin Bratt, Scott Adkins, Zara Phythian",2026-10-07T20:43:40.226Z
315635,"Action, Adventure, Science Fiction, Drama","Ma

In [0]:
from pyspark.sql import functions as F

df_creditos = spark.table("bronze.tb_credits_and_tags")

df_creditos_silver = (
    df_creditos
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.when(
            F.col("genres").isNull() | (F.trim(F.col("genres")) == "N/A"),
            None
        ).otherwise(F.trim(F.col("genres"))).alias("generos"),
        F.when(
            F.col("production_companies").isNull() | (F.trim(F.col("production_companies")) == "N/A"),
            None
        ).otherwise(F.trim(F.col("production_companies"))).alias("empresas_producao"),
        F.when(
            F.col("production_countries").isNull() | (F.trim(F.col("production_countries")) == "N/A"),
            None
        ).otherwise(F.trim(F.col("production_countries"))).alias("paises_producao"),
        F.when(
            F.col("spoken_languages").isNull() | (F.trim(F.col("spoken_languages")) == "N/A"),
            None
        ).otherwise(F.trim(F.col("spoken_languages"))).alias("idiomas_falados"),
        F.when(
            F.col("keywords").isNull() | (F.trim(F.col("keywords")) == "N/A"),
            None
        ).otherwise(F.trim(F.col("keywords"))).alias("palavras_chave"),
        F.when(
            F.col("directors").isNull() | (F.trim(F.col("directors")) == "N/A"),
            None
        ).otherwise(F.trim(F.col("directors"))).alias("diretores"),
        F.when(
            F.col("writers").isNull() | (F.trim(F.col("writers")) == "N/A"),
            None
        ).otherwise(F.trim(F.col("writers"))).alias("roteiristas"),
        F.when(
            F.col("cast").isNull() | (F.trim(F.col("cast")) == "N/A"),
            None
        ).otherwise(F.trim(F.col("cast"))).alias("elenco"),
        F.col("ingestion_datetime")
    )
)

(
    df_creditos_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_creditos_tags_filmes")
)

print("Tabela silver.tb_creditos_tags_filmes criada com sucesso!")

Tabela silver.tb_creditos_tags_filmes criada com sucesso!


In [0]:
df_reviews = spark.table("bronze.tb_movies_reviews")
df_reviews.printSchema()

root
 |-- id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- nota: double (nullable = true)
 |-- comentario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
from pyspark.sql import functions as F

df_reviews = spark.table("bronze.tb_movies_reviews")

df_reviews_silver = (
    df_reviews
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.when(
            F.trim(F.col("nome")) == "",
            None
        ).otherwise(F.trim(F.col("nome"))).alias("nome_avaliador"),
        F.col("nota").cast("double").alias("nota"),
        F.trim(F.col("comentario")).alias("comentario"),
        F.col("ingestion_datetime")
    )
)

(
    df_reviews_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_avaliacoes_filmes")
)

print("Tabela silver.tb_avaliacoes_filmes criada com sucesso!")

Tabela silver.tb_avaliacoes_filmes criada com sucesso!


In [0]:
display(
    spark.sql("SHOW TABLES IN silver")
)

database,tableName,isTemporary
silver,tb_avaliacoes_filmes,false
silver,tb_creditos_tags_filmes,false
silver,tb_financeiro_filmes,false
silver,tb_info_filmes,false
silver,tb_metricas_engajamento,false


In [0]:
from pyspark.sql import functions as F

df_creditos = spark.table("bronze.tb_credits_and_tags")

df_generos = (
    df_creditos
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.regexp_replace(F.col("genres"), ";", ",").alias("generos")
    )
    .withColumn(
        "genero",
        F.explode(F.split(F.col("generos"), ","))
    )
    .select(
        "id_filme",
        F.initcap(F.trim(F.col("genero"))).alias("nome_genero")
    )
    .filter(F.col("nome_genero").isNotNull())
    .filter(F.col("nome_genero") != "")
    .filter(~F.col("nome_genero").rlike("^[0-9]+$"))
    .dropDuplicates(["id_filme", "nome_genero"])
)

(
    df_generos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_generos")
)

print("Tabela silver.tb_generos criada com sucesso!")

Tabela silver.tb_generos criada com sucesso!


In [0]:
from pyspark.sql import functions as F

df_creditos = spark.table("bronze.tb_credits_and_tags")

df_atores = (
    df_creditos
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.explode(F.split(F.regexp_replace(F.col("cast"), ";", ","), ",")).alias("nome")
    )
    .withColumn("tipo_entidade", F.lit("Ator"))
)

df_diretores = (
    df_creditos
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.explode(F.split(F.regexp_replace(F.col("directors"), ";", ","), ",")).alias("nome")
    )
    .withColumn("tipo_entidade", F.lit("Diretor"))
)

df_roteiristas = (
    df_creditos
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.explode(F.split(F.regexp_replace(F.col("writers"), ";", ","), ",")).alias("nome")
    )
    .withColumn("tipo_entidade", F.lit("Roteirista"))
)

df_produtoras = (
    df_creditos
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.explode(F.split(F.regexp_replace(F.col("production_companies"), ";", ","), ",")).alias("nome")
    )
    .withColumn("tipo_entidade", F.lit("Produtora"))
)

df_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
    .select(
        "id_filme",
        F.initcap(F.trim(F.col("nome"))).alias("nome_entidade"),
        "tipo_entidade"
    )
    .filter(F.col("nome_entidade").isNotNull())
    .filter(F.col("nome_entidade") != "")
    .filter(~F.upper(F.col("nome_entidade")).isin("N/A", "NULL"))
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

(
    df_pessoas_empresas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_pessoas_empresas")
)

print("Tabela silver.tb_pessoas_empresas criada com sucesso!")

Tabela silver.tb_pessoas_empresas criada com sucesso!


In [0]:
from pyspark.sql import functions as F

df_reviews = spark.table("bronze.tb_movies_reviews")

df_avaliacoes_usuarios = (
    df_reviews
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.trim(F.col("nome")).alias("nome_usuario"),
        
        # Nota válida somente entre 0 e 10
        F.when(
            (F.col("nota") >= 0) & (F.col("nota") <= 10),
            F.col("nota").cast("double")
        ).otherwise(None).alias("nota_usuario"),
        
        # Comentário vazio ou apenas espaços → "Sem comentário"
        F.when(
            F.col("comentario").isNull() |
            (F.trim(F.col("comentario")) == ""),
            F.lit("Sem comentário")
        ).otherwise(
            F.trim(F.col("comentario"))
        ).alias("comentario_usuario")
    )
    # Remove avaliações integralmente duplicadas
    .dropDuplicates([
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    ])
)

(
    df_avaliacoes_usuarios.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_avaliacoes_usuarios")
)

print("Tabela silver.tb_avaliacoes_usuarios criada com sucesso!")

Tabela silver.tb_avaliacoes_usuarios criada com sucesso!


In [0]:
spark.sql("SHOW TABLES IN bronze").show(truncate=False)

+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|bronze  |tb_credits_and_tags |false      |
|bronze  |tb_movies_financials|false      |
|bronze  |tb_movies_info      |false      |
|bronze  |tb_movies_metrics   |false      |
|bronze  |tb_movies_reviews   |false      |
+--------+--------------------+-----------+



In [0]:
dbutils.widgets.text("data_inicio", "", "Data início (MM-DD-YYYY)")
dbutils.widgets.text("data_fim", "", "Data fim (MM-DD-YYYY)")

print("Widgets criados com sucesso!")

Widgets criados com sucesso!


In [0]:
import requests
from pyspark.sql import functions as F

# Recupera as datas informadas nos widgets
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

# Verifica se as datas foram preenchidas
if not data_inicio or not data_fim:
    raise ValueError(
        "Preencha os widgets 'data_inicio' e 'data_fim' no formato MM-DD-YYYY."
    )

# Endpoint da API do Banco Central
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra"
    "&$format=json"
)

# Consulta a API
resposta = requests.get(url)

if resposta.status_code != 200:
    raise Exception(
        f"Erro ao consultar a API do Banco Central: {resposta.status_code}"
    )

dados = resposta.json()["value"]

# Verifica se houve retorno
if not dados:
    raise ValueError(
        "A API não retornou cotações para o período informado. "
        "Verifique as datas ou escolha um período maior."
    )

# Cria DataFrame Spark
df_cotacao = spark.createDataFrame(dados)

# Tratamento das colunas
df_cotacao = (
    df_cotacao
    .withColumn(
        "data_cotacao",
        F.to_date(F.col("dataHoraCotacao"))
    )
    .withColumn(
        "cotacao_compra",
        F.col("cotacaoCompra").cast("decimal(18,6)")
    )
    .select(
        "data_cotacao",
        "cotacao_compra"
    )
)

# Salva na camada Bronze
(
    df_cotacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.tb_cotacao_dolar")
)

print("Tabela bronze.tb_cotacao_dolar criada com sucesso!")
print(f"Quantidade de cotações obtidas: {df_cotacao.count()}")

Tabela bronze.tb_cotacao_dolar criada com sucesso!
Quantidade de cotações obtidas: 7


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Lê a cotação da camada Bronze
df_cotacao = spark.table("bronze.tb_cotacao_dolar")

# Define o primeiro e o último dia disponíveis
limites = (
    df_cotacao
    .select(
        F.min("data_cotacao").alias("data_min"),
        F.max("data_cotacao").alias("data_max")
    )
    .collect()[0]
)

data_min = limites["data_min"]
data_max = limites["data_max"]

# Cria uma sequência contínua de datas
df_datas = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(data_min),
                F.lit(data_max),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("data_cotacao")
    )
)

# Junta todas as datas com as cotações existentes
df_cotacao_completa = (
    df_datas
    .join(df_cotacao, on="data_cotacao", how="left")
)

# Forward Fill:
# Preenche dias sem cotação com o último valor disponível
janela = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_cotacao_silver = (
    df_cotacao_completa
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(janela)
    )
    .select(
        "data_cotacao",
        "cotacao_compra"
    )
    .orderBy("data_cotacao")
)

# Salva na camada Silver
(
    df_cotacao_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_cotacao_dolar")
)

print("Tabela silver.tb_cotacao_dolar criada com sucesso!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela silver.tb_cotacao_dolar criada com sucesso!


In [0]:
from pyspark.sql import functions as F

# Cria o schema Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# Lê a dimensão de filmes da Silver
df_info = spark.table("silver.tb_info_filmes")

# Cria a dimensão de filmes
df_dim_movies = (
    df_info
    .select(
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        F.col("status").alias("status_filme"),
        "sinopse"
    )
    .dropDuplicates(["id_filme"])
)

# Gera a chave substituta
df_dim_movies = (
    df_dim_movies
    .withColumn(
        "sk_movie_id",
        F.monotonically_increasing_id()
    )
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

# Salva a dimensão
(
    df_dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_movies")
)

print("Tabela gold.dim_movies criada com sucesso!")

Tabela gold.dim_movies criada com sucesso!


In [0]:
from pyspark.sql import functions as F

# Lê os gêneros da camada Silver
df_generos = spark.table("silver.tb_generos")

# Cria a dimensão de gêneros
df_dim_genres = (
    df_generos
    .select(
        F.trim(F.col("nome_genero")).alias("nome_genero")
    )
    .filter(F.col("nome_genero").isNotNull())
    .filter(F.col("nome_genero") != "")
    .dropDuplicates(["nome_genero"])
)

# Gera a chave substituta
df_dim_genres = (
    df_dim_genres
    .withColumn(
        "sk_genre_id",
        F.monotonically_increasing_id()
    )
    .select(
        "sk_genre_id",
        "nome_genero"
    )
)

# Salva a dimensão
(
    df_dim_genres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_genres")
)

print("Tabela gold.dim_genres criada com sucesso!")

Tabela gold.dim_genres criada com sucesso!


In [0]:
from pyspark.sql import functions as F

# Lê pessoas e empresas da camada Silver
df_pessoas_empresas = spark.table("silver.tb_pessoas_empresas")

# Filtra somente os tipos que pertencem à dimensão de pessoas
df_dim_people = (
    df_pessoas_empresas
    .filter(
        F.col("tipo_entidade").isin(
            "Ator",
            "Diretor",
            "Roteirista"
        )
    )
    .select(
        F.trim(F.col("nome_entidade")).alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .filter(F.col("nome_pessoa").isNotNull())
    .filter(F.col("nome_pessoa") != "")
    .dropDuplicates([
        "nome_pessoa",
        "tipo_pessoa"
    ])
)

# Gera a chave substituta
df_dim_people = (
    df_dim_people
    .withColumn(
        "sk_person_id",
        F.monotonically_increasing_id()
    )
    .select(
        "sk_person_id",
        "nome_pessoa",
        "tipo_pessoa"
    )
)

# Salva a dimensão
(
    df_dim_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_people")
)

print("Tabela gold.dim_people criada com sucesso!")

Tabela gold.dim_people criada com sucesso!


In [0]:
from pyspark.sql import functions as F

# Lê pessoas e empresas da camada Silver
df_pessoas_empresas = spark.table("silver.tb_pessoas_empresas")

# Filtra somente as produtoras
df_dim_companies = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        F.trim(F.col("nome_entidade")).alias("nome_produtora")
    )
    .filter(F.col("nome_produtora").isNotNull())
    .filter(F.col("nome_produtora") != "")
    .dropDuplicates(["nome_produtora"])
)

# Gera a chave substituta
df_dim_companies = (
    df_dim_companies
    .withColumn(
        "sk_company_id",
        F.monotonically_increasing_id()
    )
    .select(
        "sk_company_id",
        "nome_produtora"
    )
)

# Salva a dimensão
(
    df_dim_companies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_companies")
)

print("Tabela gold.dim_companies criada com sucesso!")

Tabela gold.dim_companies criada com sucesso!


In [0]:
from pyspark.sql import functions as F

# Avaliações da camada Silver
df_avaliacoes = spark.table("silver.tb_avaliacoes_usuarios")

# Dimensão de filmes
df_movies = spark.table("gold.dim_movies")

# Resume as avaliações por filme
df_reviews_resumo = (
    df_avaliacoes
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(
            F.avg("nota_usuario"),
            2
        ).alias("nota_media_usuarios")
    )
)

# Relaciona cada avaliação ao filme através do id_filme
df_dim_reviews = (
    df_reviews_resumo
    .join(
        df_movies.select(
            "sk_movie_id",
            "id_filme"
        ),
        on="id_filme",
        how="inner"
    )
    .withColumn(
        "sk_review_id",
        F.monotonically_increasing_id()
    )
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

# Salva a dimensão
(
    df_dim_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_reviews")
)

print("Tabela gold.dim_reviews criada com sucesso!")

Tabela gold.dim_reviews criada com sucesso!


In [0]:
from pyspark.sql import functions as F

# Dimensão de filmes
df_movies = spark.table("gold.dim_movies")

# Métricas financeiras
df_financeiro = spark.table("silver.tb_financeiro_filmes")

# Métricas de engajamento
df_metricas = spark.table("silver.tb_metricas_engajamento")

# Junta as informações por filme
df_fact = (
    df_movies
    .join(
        df_financeiro,
        on="id_filme",
        how="left"
    )
    .join(
        df_metricas,
        on="id_filme",
        how="left"
    )
)

# Somente filmes lançados
df_fact = (
    df_fact
    .filter(
        F.upper(F.trim(F.col("status_filme"))) == "LANÇADO"
    )
)

# Seleciona e tipa as métricas da tabela fato
df_fact_movies_performance = (
    df_fact
    .select(
        F.col("sk_movie_id").cast("bigint").alias("sk_movie_id"),

        F.col("orcamento").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro").cast("decimal(18,2)").alias("lucro_usd"),

        F.lit(None).cast("decimal(18,2)").alias("orcamento_brl"),
        F.lit(None).cast("decimal(18,2)").alias("receita_brl"),
        F.lit(None).cast("decimal(18,2)").alias("lucro_brl"),

        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.col("quantidade_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.col("quantidade_votos_imdb").cast("int").alias("qtd_votos_imdb")
    )
    .dropDuplicates(["sk_movie_id"])
)

# Salva a tabela fato
(
    df_fact_movies_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_movies_performance")
)

print("Tabela gold.fact_movies_performance criada com sucesso!")

Tabela gold.fact_movies_performance criada com sucesso!


In [0]:
from pyspark.sql import functions as F

# Dimensão de filmes
df_movies = spark.table("gold.dim_movies")

# Métricas financeiras
df_financeiro = spark.table("silver.tb_financeiro_filmes")

# Métricas de engajamento
df_metricas = spark.table("silver.tb_metricas_engajamento")

# Junta as informações por filme
df_fact = (
    df_movies
    .join(
        df_financeiro,
        on="id_filme",
        how="left"
    )
    .join(
        df_metricas,
        on="id_filme",
        how="left"
    )
)

# Somente filmes lançados
df_fact = (
    df_fact
    .filter(
        F.upper(F.trim(F.col("status_filme"))) == "LANÇADO"
    )
)

# Seleciona e tipa as métricas da tabela fato
df_fact_movies_performance = (
    df_fact
    .select(
        F.col("sk_movie_id").cast("bigint").alias("sk_movie_id"),

        F.col("orcamento").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro").cast("decimal(18,2)").alias("lucro_usd"),

        F.lit(None).cast("decimal(18,2)").alias("orcamento_brl"),
        F.lit(None).cast("decimal(18,2)").alias("receita_brl"),
        F.lit(None).cast("decimal(18,2)").alias("lucro_brl"),

        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.col("quantidade_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.col("quantidade_votos_imdb").cast("int").alias("qtd_votos_imdb")
    )
    .dropDuplicates(["sk_movie_id"])
)

# Salva a tabela fato
(
    df_fact_movies_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_movies_performance")
)

print("Tabela gold.fact_movies_performance criada com sucesso!")

Tabela gold.fact_movies_performance criada com sucesso!
